In [ ]:
import sys,subprocess,importlib.util
if importlib.util.find_spec("mne") is None: subprocess.check_call([sys.executable,"-m","pip","install","-q","mne"])
import os,glob,zipfile,numpy as np,pandas as pd,matplotlib.pyplot as plt,mne
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score,classification_report
np.random.seed(42)


In [ ]:
def summation_unit(x,w): return float(np.dot(x,w[1:])+w[0])
def activation(z,name="step"):
    if name=="step": return 1.0 if z>=0 else 0.0
    if name=="bipolar": return 1.0 if z>=0 else -1.0
    if name=="sigmoid": return 1/(1+np.exp(-np.clip(z,-60,60)))
    if name=="tanh": return np.tanh(z)
    if name=="relu": return max(0.0,z)
    if name=="leaky_relu": return z if z>=0 else 0.01*z
    raise ValueError(name)
def derivative(z,name):
    if name=="sigmoid":
        a=activation(z,"sigmoid"); return a*(1-a)
    if name=="tanh":
        a=np.tanh(z); return 1-a*a
    if name=="relu": return 1.0 if z>0 else 0.0
    if name=="leaky_relu": return 1.0 if z>=0 else 0.01
    return 1.0
def comparator(t,o): return float(t-o)
def train_single_neuron(X,y,act="step",lr=0.05,w0=None,tol=0.002,max_epochs=1000):
    w=np.array([10.0,0.2,-0.75] if w0 is None else w0,dtype=float); hist=[]
    for e in range(1,max_epochs+1):
        for xi,yi in zip(X,y):
            z=summation_unit(xi,w); o=activation(z,act); err=comparator(yi,o)
            g=err if act in ("step","bipolar") else err*derivative(z,act)
            w[0]+=lr*g; w[1:]+=lr*g*xi
        p=np.array([activation(summation_unit(xi,w),act) for xi in X],float)
        s=float(np.sum((y-p)**2)); hist.append(s)
        if s<=tol: break
    return w,e,np.array(hist),p
def predict_two_layer(X,W1,W2):
    H=1/(1+np.exp(-np.clip(np.c_[np.ones(len(X)),X]@W1,-60,60)))
    return 1/(1+np.exp(-np.clip(np.c_[np.ones(len(X)),H]@W2,-60,60)))
def train_two_layer(X,y,hidden=2,outputs=1,lr=0.05,tol=0.002,max_epochs=1000,seed=42):
    r=np.random.default_rng(seed); W1=r.uniform(-.05,.05,(X.shape[1]+1,hidden)); W2=r.uniform(-.05,.05,(hidden+1,outputs)); Y=np.asarray(y,float).reshape(len(X),outputs); hist=[]
    for e in range(1,max_epochs+1):
        for xi,yi in zip(X,Y):
            xb=np.r_[1.,xi]; h=1/(1+np.exp(-np.clip(xb@W1,-60,60))); hb=np.r_[1.,h]; o=1/(1+np.exp(-np.clip(hb@W2,-60,60)))
            do=(yi-o)*o*(1-o); dh=h*(1-h)*(W2[1:]@do); W2+=lr*np.outer(hb,do); W1+=lr*np.outer(xb,dh)
        P=predict_two_layer(X,W1,W2); s=float(np.sum((Y-P)**2)); hist.append(s)
        if s<=tol: break
    return W1,W2,e,np.array(hist)
def pseudo_inverse_classifier(X,y):
    Xb=np.c_[np.ones(len(X)),X]; w=np.linalg.pinv(Xb)@y; return w,(Xb@w>=.5).astype(int)
def extract_zip_archives(root="/content"):
    out=os.path.join(root,"sleep_edf_extracted"); os.makedirs(out,exist_ok=True)
    for z in glob.glob(os.path.join(root,"**","*.zip"),recursive=True):
        if zipfile.is_zipfile(z):
            d=os.path.join(out,os.path.splitext(os.path.basename(z))[0]); os.makedirs(d,exist_ok=True)
            with zipfile.ZipFile(z) as f: f.extractall(d)
def find_pair(root="/content"):
    extract_zip_archives(root)
    fs=[p for p in glob.glob(os.path.join(root,"**","*"),recursive=True) if os.path.isfile(p) and p.lower().endswith(".edf")]
    ps=sorted([p for p in fs if "psg" in os.path.basename(p).lower()]); hs=sorted([p for p in fs if "hypnogram" in os.path.basename(p).lower()])
    for p in ps:
        pref=os.path.basename(p)[:6].lower(); m=[h for h in hs if os.path.basename(h).lower().startswith(pref)]
        if m: return p,m[0]
    raise FileNotFoundError("Upload matching PSG and Hypnogram EDF files to /content")
def band_power(x,sf,a,b):
    f=np.fft.rfftfreq(len(x),1/sf); p=np.abs(np.fft.rfft(x))**2; m=(f>=a)&(f<b); return float(p[m].mean()) if np.any(m) else 0.0
def load_project():
    psg,hyp=find_pair(); raw=mne.io.read_raw_edf(psg,preload=True,verbose=False); ann=mne.read_annotations(hyp); raw.set_annotations(ann,emit_warning=False)
    ch=["EEG Fpz-Cz","EEG Pz-Oz","EOG horizontal","Resp oro-nasal","EMG submental"]; data=raw.get_data(picks=ch); sf=float(raw.info["sfreq"]); n=int(30*sf); X=[]; y=[]
    for onset,dur,label in zip(ann.onset,ann.duration,ann.description):
        if label not in ("Sleep stage 2","Sleep stage 3"): continue
        for i in range(int(dur//30)):
            s=int((onset+30*i)*sf); e=s+n
            if e>data.shape[1]: continue
            f=[]
            for j in range(5):
                v=data[j,s:e]; f += [v.mean(),v.std(),v.min(),v.max(),np.ptp(v),np.sqrt(np.mean(v*v))]
                if j<2: f += [band_power(v,sf,.5,4),band_power(v,sf,4,8),band_power(v,sf,8,13),band_power(v,sf,13,30)]
            X.append(f); y.append(0 if label=="Sleep stage 2" else 1)
    X=np.asarray(X,float); y=np.asarray(y,int); X[~np.isfinite(X)]=np.nan; med=np.nanmedian(X,axis=0); r,c=np.where(np.isnan(X)); X[r,c]=med[c]
    return X,y


In [ ]:
Xg=np.array([[0,0],[0,1],[1,0],[1,1]],float); ya=np.array([0,0,0,1],float); yx=np.array([0,1,1,0],float)
for gate,y in [("AND",ya),("XOR",yx)]:
    rows=[]
    for a in ["step","bipolar","sigmoid","relu"]:
        t=np.where(y==0,-1,1) if a=="bipolar" else y
        w,e,h,p=train_single_neuron(Xg,t,a)
        rows.append([gate,a,e,h[-1]])
    display(pd.DataFrame(rows,columns=["Gate","Activation","Epochs","Final SSE"]))


In [ ]:
rows=[]
for lr in np.arange(.1,1.01,.1):
    w,e,h,p=train_single_neuron(Xg,ya,"step",lr=lr); rows.append([lr,e,h[-1]])
d=pd.DataFrame(rows,columns=["Learning Rate","Epochs","Final SSE"]); display(d)
plt.plot(d["Learning Rate"],d["Epochs"],marker="o"); plt.xlabel("Learning Rate"); plt.ylabel("Epochs"); plt.show()


In [ ]:
C=np.array([[20,6,2,386],[16,3,6,289],[27,6,2,393],[19,1,2,110],[24,4,2,280],[22,1,5,167],[15,4,2,271],[18,4,2,274],[21,1,4,148],[16,2,4,198]],float)
cy=np.array([1,1,1,0,1,0,1,1,0,0],float); mu=C.mean(0); sd=C.std(0); CX=(C-mu)/sd
w,e,h,p=train_single_neuron(CX,cy,"sigmoid",lr=.1,w0=np.zeros(5)); pw,pp=pseudo_inverse_classifier(CX,cy)
display(pd.DataFrame({"Actual":cy.astype(int),"Perceptron":(p>=.5).astype(int),"PseudoInverse":pp}))


In [ ]:
for gate,y in [("AND",ya),("XOR",yx)]:
    W1,W2,e,h=train_two_layer(Xg,y,2,1,.05); pr=(predict_two_layer(Xg,W1,W2).ravel()>=.5).astype(int)
    print(gate,e,h[-1],pr); plt.plot(h,label=gate)
plt.xlabel("Epoch"); plt.ylabel("SSE"); plt.legend(); plt.show()


In [ ]:
for gate,Y in [("AND",np.array([[1,0],[1,0],[1,0],[0,1]],float)),("XOR",np.array([[1,0],[0,1],[0,1],[1,0]],float))]:
    W1,W2,e,h=train_two_layer(Xg,Y,2,2,.05); print(gate,e,h[-1],np.argmax(predict_two_layer(Xg,W1,W2),axis=1))


In [ ]:
for gate,y in [("AND",ya.astype(int)),("XOR",yx.astype(int))]:
    m=MLPClassifier(hidden_layer_sizes=(2,),activation="logistic",solver="lbfgs",random_state=42,max_iter=5000); m.fit(Xg,y); print(gate,m.predict(Xg),m.score(Xg,y))


In [ ]:
X,y=load_project(); Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.3,stratify=y,random_state=42)
mu=Xtr.mean(0); sd=Xtr.std(0); sd[sd==0]=1; Xtr=(Xtr-mu)/sd; Xte=(Xte-mu)/sd
m=MLPClassifier(hidden_layer_sizes=(32,16),activation="relu",max_iter=1000,random_state=42); m.fit(Xtr,ytr); p=m.predict(Xte)
print("Shape:",X.shape); print("Accuracy:",accuracy_score(yte,p)); print(classification_report(yte,p,target_names=["N2","N3"],zero_division=0))
